# 🚀 Level 5: Simulation, Monte Carlo Modelling, and Optimization

## HydroSense-Kenya — From Computation to Decision Intelligence

---

This is the core computational level. We build:

1. **Deterministic simulation** — 30-day soil moisture via Euler and RK4
2. **Stochastic simulation** — 1000+ Monte Carlo scenarios
3. **Risk quantification** — P(shortage), expected and worst-case demand
4. **Constrained optimization** — Irrigation schedule minimising water use
5. **Tradeoff analysis** — Pareto frontier

In [ ]:
import sys, os
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.join('..', 'src'))
from simulation import (
    compute_et, simulate_water_balance,
    monte_carlo_rainfall, monte_carlo_simulation,
    compute_risk_metrics, fit_rainfall_gamma,
)
from optimization import (
    gradient_descent_irrigation, pareto_tradeoff_analysis,
)
from visualization import (
    setup_publication_style, COLORS, ZONE_COLORS,
    plot_simulation_ensemble, plot_risk_histogram,
    plot_euler_vs_rk4, plot_optimization_convergence,
    plot_pareto_frontier, plot_irrigation_schedule,
)

setup_publication_style()

weather = pd.read_csv('../data/processed/cleaned_irrigation_dataset.csv')
weather['date'] = pd.to_datetime(weather['date'])
params = pd.read_csv('../data/raw/crop_zone_parameters.csv')

T = weather['temperature_c'].values
W = weather['wind_speed_mps'].values
S = weather['solar_index'].values
H = weather['humidity_pct'].values
R = weather['rainfall_mm'].values
et_daily = compute_et(T, W, S, H)
n_days = len(weather)

print(f'Level 5: Simulation & Optimization — {n_days} days loaded ✓')

---

## 1. Deterministic Soil Moisture Simulation

### Euler vs Runge-Kutta Comparison

In [ ]:
zone_results = {}

fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

for idx, (_, zp) in enumerate(params.iterrows()):
    zone = zp['zone_id']
    s0 = 30.0
    fc = zp['field_capacity_pct']
    dc = zp['drainage_coefficient']
    no_irr = np.zeros(n_days)

    euler_result = simulate_water_balance(n_days, s0, R, no_irr, et_daily, fc, dc, method='euler')
    rk4_result = simulate_water_balance(n_days, s0, R, no_irr, et_daily, fc, dc, method='rk4')

    zone_results[zone] = {'euler': euler_result, 'rk4': rk4_result, 'params': zp}

    ax = axes[idx]
    time_days = euler_result.time_days
    ax.plot(time_days, euler_result.soil_moisture, color=COLORS['orange'],
            linewidth=2, marker='o', markersize=3, label='Euler', alpha=0.8)
    ax.plot(time_days, rk4_result.soil_moisture, color=COLORS['blue'],
            linewidth=2, marker='s', markersize=3, label='RK4')
    ax.axhline(zp['min_moisture_pct'], color=COLORS['red'], linestyle='--',
               linewidth=2, alpha=0.7, label=f"Stress ({zp['min_moisture_pct']}%)")
    ax.axhline(zp['field_capacity_pct'], color=COLORS['green'], linestyle=':',
               linewidth=1.5, alpha=0.5, label=f"Field cap ({zp['field_capacity_pct']}%)")
    ax.fill_between(time_days, 0, zp['min_moisture_pct'], alpha=0.05, color=COLORS['red'])
    ax.set_ylabel('Moisture [% vol.]')
    ax.set_title(f"{zone} — {zp['crop_type'].title()}", fontweight='bold', loc='left')
    ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Day')
fig.suptitle('30-Day Soil Moisture: Euler vs RK4 (No Irrigation)',
             fontsize=14, fontweight='bold', y=1.01)
fig.tight_layout()
plt.show()

print('\nEuler vs RK4 Comparison:')
print(f"{'Zone':<10} {'Max |diff|':<15} {'Mean |diff|':<15}")
print('-' * 40)
for zone, res in zone_results.items():
    diff = np.abs(res['euler'].soil_moisture - res['rk4'].soil_moisture)
    print(f'{zone:<10} {np.max(diff):<15.4f} {np.mean(diff):<15.4f}')

---

## 2. Monte Carlo Rainfall Uncertainty Simulation

In [ ]:
observed_rain = R.copy()
observed_rain[np.isnan(observed_rain)] = 0

wet_fraction = np.mean(observed_rain > 0)
shape, scale = fit_rainfall_gamma(observed_rain)

print('Rainfall Distribution Fit')
print('=' * 50)
print(f'  Wet day probability  : {wet_fraction:.2%}')
print(f'  Gamma shape (k)      : {shape:.3f}')
print(f'  Gamma scale (θ)      : {scale:.3f}')
print(f'  Gamma mean           : {shape * scale:.2f} mm (wet days only)')

n_scenarios = 1500
rainfall_scenarios = monte_carlo_rainfall(observed_rain, n_scenarios=n_scenarios,
                                           n_days=n_days, seed=42)
print(f'\nGenerated {n_scenarios} rainfall scenarios')
print(f'  Mean total per scenario: {rainfall_scenarios.sum(axis=1).mean():.1f} mm')
print(f'  5th percentile: {np.percentile(rainfall_scenarios.sum(axis=1), 5):.1f} mm')
print(f'  95th percentile: {np.percentile(rainfall_scenarios.sum(axis=1), 95):.1f} mm')

In [ ]:
# Monte Carlo ensemble for Zone A
zp_a = params[params['zone_id'] == 'Zone_A'].iloc[0]

trajectories_a = monte_carlo_simulation(
    n_scenarios=n_scenarios, n_days=n_days, s0=30.0,
    rainfall_scenarios=rainfall_scenarios,
    irrigation=np.zeros(n_days),
    temperature=T, wind_speed=W, solar_index=S, humidity=H,
    field_capacity=zp_a['field_capacity_pct'],
    drainage_coeff=zp_a['drainage_coefficient'],
    method='rk4',
)

time_days = np.arange(n_days + 1)
fig = plot_simulation_ensemble(
    trajectories_a, time_days,
    min_moisture=zp_a['min_moisture_pct'],
    field_capacity=zp_a['field_capacity_pct'],
    title=f'Monte Carlo Ensemble — Zone A (Tomato) | {n_scenarios} Scenarios',
)
plt.show()

shortage_pct = np.mean(trajectories_a.min(axis=1) < zp_a['min_moisture_pct'])
print(f'📊 Without irrigation, {shortage_pct:.0%} of scenarios result in moisture stress')

---

## 3. Risk Quantification

In [ ]:
print('=' * 70)
print('RISK ASSESSMENT — All Zones (No Irrigation Baseline)')
print('=' * 70)

for _, zp in params.iterrows():
    zone = zp['zone_id']
    trajectories = monte_carlo_simulation(
        n_scenarios=n_scenarios, n_days=n_days, s0=30.0,
        rainfall_scenarios=rainfall_scenarios,
        irrigation=np.zeros(n_days),
        temperature=T, wind_speed=W, solar_index=S, humidity=H,
        field_capacity=zp['field_capacity_pct'],
        drainage_coeff=zp['drainage_coefficient'],
    )
    metrics = compute_risk_metrics(
        trajectories, zp['min_moisture_pct'], zp['field_capacity_pct'])
    print(f"\n--- {zone} ({zp['crop_type']}) ---")
    print(f'  P(water shortage)        : {metrics.p_shortage:.1%}')
    print(f'  P(over-irrigation)       : {metrics.p_over_irrigation:.1%}')
    print(f'  Expected demand          : {metrics.expected_demand:.1f} mm')
    print(f'  Worst-case demand (95%)  : {metrics.worst_case_demand:.1f} mm')
    print(f'  Mean min moisture        : {metrics.mean_min_moisture:.1f}%')
    print(f'  Avg shortage days        : {metrics.shortage_days_mean:.1f} days')

fig = plot_risk_histogram(trajectories_a, min_moisture=zp_a['min_moisture_pct'])
plt.show()

---

## 4. Irrigation Optimization

Minimize total irrigation subject to $S_t \geq S_{\min}$ for all $t$:

$$\mathcal{L}(I) = \sum I_t + \lambda \sum \max(0, S_{\min} - S_t)^2$$

In [ ]:
print('Optimizing irrigation schedule for Zone A (tomato)...')
print(f"  Objective: minimize water use, S(t) >= {zp_a['min_moisture_pct']}% at all times")
print()

opt_result = gradient_descent_irrigation(
    n_days=n_days, s0=30.0,
    rainfall=R, et=et_daily,
    field_capacity=zp_a['field_capacity_pct'],
    drainage_coeff=zp_a['drainage_coefficient'],
    min_moisture=zp_a['min_moisture_pct'],
    penalty_weight=200.0,
    max_iter=250,
)

print(f'  Converged: {opt_result.converged}')
print(f'  Iterations: {opt_result.iterations}')
print(f'  Total water used: {opt_result.total_water_used:.2f} mm')
print(f'  Daily average: {opt_result.total_water_used/n_days:.2f} mm/day')
print(f'  Constraint violation: {opt_result.constraint_violation:.4f}')
print(f'  Min moisture achieved: {opt_result.final_moisture.min():.1f}%')

fig = plot_optimization_convergence(opt_result.objective_history)
plt.show()

In [ ]:
fig = plot_irrigation_schedule(
    opt_result.irrigation_schedule,
    opt_result.final_moisture,
    min_moisture=zp_a['min_moisture_pct'],
)
plt.show()

print('Optimized Irrigation Schedule — Zone A')
print('=' * 60)
print(f"{'Day':<6} {'Rain':<10} {'ET':<10} {'Irrig':<10} {'S(t)':<10}")
print('-' * 46)
for d in range(n_days):
    print(f"{d+1:<6} {R[d]:<10.1f} {et_daily[d]:<10.2f} "
          f"{opt_result.irrigation_schedule[d]:<10.2f} "
          f"{opt_result.final_moisture[d]:<10.1f}")

---

## 5. Tradeoff Analysis: Water Conservation vs Crop Safety

In [ ]:
print('Computing Pareto frontier...')
pareto_results = pareto_tradeoff_analysis(
    n_days=n_days, s0=30.0,
    rainfall=R, et=et_daily,
    field_capacity=zp_a['field_capacity_pct'],
    drainage_coeff=zp_a['drainage_coefficient'],
    min_moisture=zp_a['min_moisture_pct'],
    lambda_values=np.array([1, 5, 10, 25, 50, 100, 250, 500]),
    max_iter=150,
)

fig = plot_pareto_frontier(pareto_results)
plt.show()

print('\nPareto Frontier — Zone A:')
print(f"{'λ':<8} {'Water (mm)':<14} {'Violation':<14} {'Converged':<10}")
print('-' * 46)
for r in pareto_results:
    print(f"{r['lambda']:<8.0f} {r['total_water_mm']:<14.1f} "
          f"{r['constraint_violation']:<14.3f} {'✓' if r['converged'] else '✗':<10}")

---

## 6. Summary and Recommendations

### Key Results

- Monte Carlo with 1500 scenarios reveals high crop stress risk without irrigation
- RK4 and Euler agree closely at daily time steps
- Optimized irrigation reduces water use while maintaining moisture above threshold
- Pareto frontier shows the water-conservation vs crop-safety tradeoff

### Recommendations

1. **Use RK4** for all production simulations
2. **Run Monte Carlo with ≥1000 scenarios**
3. **Select λ = 100–250** for operational scheduling
4. **Zone C (maize) is the priority** — highest shortage risk

**Next:** Level 6 integrates all components and validates.